# Implementing an Earley Parser

F# counterpart of `ANTLR4-Python/Earley-Parser/Earley-Parser.ipynb`. A real FsLexYacc grammar-for-grammars parser replaces the generated Python ANTLR classes. Earley items use F# structural equality and ordering; no monkey-patching or custom string-based hash is needed.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#r "../../Chapter-08/GrammarSyntax/bin/Debug/net9.0/FsLexYacc.Runtime.dll"
#r "../../Chapter-08/GrammarSyntax/bin/Debug/net9.0/GrammarSyntax.dll"
#load "../../Chapter-08/EarleyPure.fsx"
open GrammarSyntax
open EarleyPure
open System.IO

## A grammar for grammars

Nonterminals and terminals are different union cases. The generated parser expands alternatives into typed rules and verifies that each referenced nonterminal is defined. Epsilon productions have an empty body.

In [ ]:
let grammar = GrammarSyntax.Parse.file "simple.g"
printfn "%s\n%A" (File.ReadAllText "simple.g") grammar

## Earley’s algorithm

An item stores a rule, a dot position, and its origin column. Prediction adds initial items for a nonterminal; scanning advances over a matching token; completion advances callers from the origin chart. Closure repeats until no new item is added, including nullable productions. These are the canonical F# definitions from Chapter 08:

```fsharp
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"

let parse = Parse.file

type EarleyItem = { Rule: Rule; Dot: int; Origin: int }
type Token = { Kind: string; Lexeme: string }

let isComplete (item: EarleyItem) = item.Dot = item.Rule.Body.Length
let symbolAfterDot (item: EarleyItem) = List.tryItem item.Dot item.Rule.Body
let sameVar name item = symbolAfterDot item = Some(Nonterminal name)
let scanToken token item = symbolAfterDot item = Some(Terminal token.Kind)
let nextVar item =
    match symbolAfterDot item with Some(Nonterminal name) -> Some name | _ -> None
let moveDot (item: EarleyItem) = { item with Dot = item.Dot + 1 }
let startRule grammar = { Head = "ŝ"; Body = [Nonterminal grammar.Start] }
let startItem grammar = { Rule = startRule grammar; Dot = 0; Origin = 0 }
let finishItem grammar = { Rule = startRule grammar; Dot = 1; Origin = 0 }

// Prediction and completion only enlarge a chart set.
let predict grammar (chart: Set<EarleyItem> array) position =
    let additions =
        chart[position]
        |> Seq.choose nextVar
        |> Seq.collect (fun name ->
            grammar.Rules
            |> Seq.filter (fun rule -> rule.Head = name)
            |> Seq.map (fun rule -> { Rule = rule; Dot = 0; Origin = position }))
        |> Set.ofSeq
    chart[position] <- Set.union chart[position] additions

let complete (chart: Set<EarleyItem> array) position =
    let additions =
        chart[position]
        |> Seq.filter isComplete
        |> Seq.collect (fun item ->
            chart[item.Origin]
            |> Seq.filter (sameVar item.Rule.Head)
            |> Seq.map moveDot)
        |> Set.ofSeq
    chart[position] <- Set.union chart[position] additions

let scan (tokens: Token array) (chart: Set<EarleyItem> array) position =
    if position < tokens.Length then
        chart[position + 1] <-
            chart[position]
            |> Seq.filter (scanToken tokens[position])
            |> Seq.map moveDot
            |> Set.ofSeq

let earleyParse grammar tokens =
    let input = List.toArray tokens
    let chart = Array.create (input.Length + 1) Set.empty
    chart[0] <- Set.singleton (startItem grammar)
    for position in 0 .. input.Length do
        let mutable changed = true
        while changed do
            let previous = chart[position]
            predict grammar chart position
            complete chart position
            changed <- chart[position] <> previous
        scan input chart position
    chart

let recognizes grammar tokens =
    let chart = earleyParse grammar tokens
    Set.contains (finishItem grammar) chart[tokens.Length]

open System
open System.Text.RegularExpressions

let tokenize (source: string) =
    let scanner = Regex(@"\G(?:\s+|[1-9][0-9]*|0|[a-zA-Z][a-zA-Z0-9_]*|[-+*/()])")
    let rec loop position result =
        if position = source.Length then List.rev result
        else
            let token = scanner.Match(source, position)
            if not token.Success then failwithf "Illegal character at column %d" (position + 1)
            let next = position + token.Length
            if String.IsNullOrWhiteSpace token.Value then loop next result
            else
                let kind =
                    if Char.IsDigit token.Value[0] then "NUMBER"
                    elif Char.IsLetter token.Value[0] then "IDENTIFIER"
                    else token.Value
                loop next ({ Kind = kind; Lexeme = token.Value } :: result)
    loop 0 []

let test file word =
    let grammar = parse file
    let tokens = tokenize word
    let chart = earleyParse grammar tokens
    chart |> Array.iteri (fun position items -> printfn "Q%d = %A" position items)
    let accepted = Set.contains (finishItem grammar) chart[tokens.Length]
    printfn "Accepted: %b" accepted
    accepted
```


In [ ]:
let isComplete (item: EarleyItem) = item.Dot = item.Rule.Body.Length
let sameVar variable (item: EarleyItem) =
    List.tryItem item.Dot item.Rule.Body = Some (Nonterminal variable)
let formatItem (item: EarleyItem) =
    let names = item.Rule.Body |> List.map (function Nonterminal name -> name | Terminal name -> "'" + name + "'")
    let before = names |> List.take item.Dot |> String.concat " "
    let after = names |> List.skip item.Dot |> String.concat " "
    sprintf "⟨%s → %s • %s, %d⟩" item.Rule.Head before after item.Origin

let test text =
    let tokens = tokenize text
    let chart = earleyParse grammar tokens
    for column, items in Array.indexed chart do
        printfn "Chart %d" column
        for item in items do printfn "  %s" (formatItem item)
    let accepted = recognizes grammar tokens
    printfn "Accepted: %b" accepted
    accepted

checkNotebook (test "1+2*3")
checkNotebook (test "(1+2)*3")
checkNotebook (not (test "1+2*"))
let nullable = GrammarSyntax.Parse.text "s: a a; a: | 'x';"
checkNotebook (recognizes nullable [])
checkNotebook (recognizes nullable ([{ Kind = "x"; Lexeme = "x" }; { Kind = "x"; Lexeme = "x" }] : EarleyPure.Token list))